# 40 — Cliff-Weighted LGBM + Auxiliary Cliff-Role Predictor

LGBM trained on PXR CRC data with **cliff-aware sample weighting** and an auxiliary cliff-role
classifier. Key ideas:

1. Cliff members (Tanimoto ≥ 0.6, |ΔpEC50| ≥ 1.0) are upweighted so the main regressor
   specifically learns the active/inactive distinction in cliff pairs.
2. A separate LGBM classifies cliff_role (−1, 0, +1). Its OOF probabilities are saved as
   auxiliary signal for downstream ensembles.
3. We compare overall and cliff-specific OOF RAE against the unweighted baseline.

In [1]:
import sys, os, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
try:
    sys.stdout.reconfigure(encoding="utf-8")
except AttributeError:
    pass
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge

from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices, compute_metrics
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

LGBM_PARAMS = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1
)

SEED = 42
print('Setup complete')

Setup complete


## 1. Load data + cliff labels

In [2]:
tr = load_train()
te = load_test()
print(f'Train: {len(tr):,}  Test: {len(te):,}')

# ── Try to load pre-computed cliff labels ──────────────────────────────────
cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
cliff_loaded = False
if cliff_path.exists():
    try:
        cliff_df = pd.read_parquet(cliff_path)
        # Expect columns: smiles (or inchikey), cliff_role
        cliff_loaded = True
        print(f'Loaded cliff labels from {cliff_path}')
        print(cliff_df['cliff_role'].value_counts())
    except Exception as e:
        print(f'Failed to load cliff labels: {e}. Will compute inline.')

# ── Compute cliff labels inline if parquet unavailable ────────────────────
if not cliff_loaded:
    print('Computing cliff labels inline (Tanimoto >= 0.6, |ΔpEC50| >= 1.0)...')
    from rdkit import Chem, DataStructs
    from rdkit.Chem import AllChem

    gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

    def get_fp(smi):
        try:
            mol = Chem.MolFromSmiles(smi)
            return gen.GetFingerprint(mol) if mol else None
        except Exception:
            return None

    fps = [get_fp(s) for s in tr['smiles']]
    y_vals = tr['pec50'].values
    n = len(tr)

    cliff_role = np.zeros(n, dtype=np.int8)  # 0 = not cliff member

    print('  Building pairwise cliff annotations (this may take ~1 min)...')
    for i in range(n):
        if fps[i] is None:
            continue
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], [f for f in fps if f is not None])
        # Map back — need indices of non-None fps
        break  # replaced by vectorised batch below

    # Vectorised via numpy batch
    fp_mat = morgan_fp_batch(tr['smiles'].tolist())  # (N, 2048) uint8

    BATCH = 256
    for i in range(0, n, BATCH):
        chunk = fp_mat[i:i+BATCH].astype(np.float32)           # (B, 2048)
        full  = fp_mat.astype(np.float32)                       # (N, 2048)
        inter = chunk @ full.T                                   # (B, N)
        union = (chunk.sum(1, keepdims=True) + full.sum(1)[None, :] - inter)
        with np.errstate(divide='ignore', invalid='ignore'):
            tan = np.where(union > 0, inter / union, 0.0)       # (B, N)
        for bi, gi in enumerate(range(i, min(i + BATCH, n))):
            for j in range(n):
                if gi == j:
                    continue
                if tan[bi, j] >= 0.6 and abs(y_vals[gi] - y_vals[j]) >= 1.0:
                    if y_vals[gi] > y_vals[j]:
                        cliff_role[gi] = 1   # cliff-active (higher pEC50)
                    else:
                        cliff_role[gi] = -1  # cliff-inactive (lower pEC50)

    cliff_df = pd.DataFrame({'smiles': tr['smiles'], 'cliff_role': cliff_role})
    cliff_df.to_parquet(DATA_PROCESSED / 'cliff_labels.parquet', index=False)
    print(f'Cliff labels computed and saved.')
    print(cliff_df['cliff_role'].value_counts())

# ── Merge cliff_role onto training DataFrame ───────────────────────────────
if 'smiles' in cliff_df.columns:
    tr = tr.merge(cliff_df[['smiles', 'cliff_role']], on='smiles', how='left')
else:
    # Fall back: assign by position if lengths match
    if len(cliff_df) == len(tr):
        tr['cliff_role'] = cliff_df['cliff_role'].values
    else:
        tr['cliff_role'] = 0

tr['cliff_role'] = tr['cliff_role'].fillna(0).astype(int)
print(f"\nCliff role distribution:\n{tr['cliff_role'].value_counts()}")

Train: 4,139  Test: 513
Computing cliff labels inline (Tanimoto >= 0.6, |ΔpEC50| >= 1.0)...


  Building pairwise cliff annotations (this may take ~1 min)...


Cliff labels computed and saved.
cliff_role
 0    4077
 1      34
-1      28
Name: count, dtype: int64

Cliff role distribution:
cliff_role
 0    4077
 1      34
-1      28
Name: count, dtype: int64


In [3]:
# ── Featurize ──────────────────────────────────────────────────────────────
print('Featurizing...')
X_tr = impute(combined(tr['smiles'].tolist()))
X_te = impute(combined(te['smiles'].tolist()))
y_tr = tr['pec50'].values.astype(np.float32)

scaffolds = tr['smiles'].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=5, seed=SEED)

print(f'X_tr: {X_tr.shape}  X_te: {X_te.shape}')

Featurizing...


X_tr: (4139, 2265)  X_te: (513, 2265)


## 2. Cliff-aware sample weights

In [4]:
def make_sample_weights(pec50_arr, cliff_role_arr):
    """Assign per-compound weights emphasising cliff and active compounds.

    Weight schedule:
      inactive (pEC50 < 5):                           1.0
      moderate (5 <= pEC50 < 6):                      3.0
      cliff-active (pEC50 >= 6 AND cliff_role == 1): 10.0
      cliff-inactive (cliff_role == -1):              8.0
      active non-cliff (pEC50 >= 6):                  6.0
    """
    w = np.ones(len(pec50_arr), dtype=np.float32)
    moderate  = (pec50_arr >= 5.0) & (pec50_arr < 6.0)
    active    = pec50_arr >= 6.0
    c_active  = (active) & (cliff_role_arr == 1)
    c_inact   = cliff_role_arr == -1
    act_nocliff = active & (cliff_role_arr == 0)

    w[moderate]    = 3.0
    w[act_nocliff] = 6.0
    w[c_inact]     = 8.0
    w[c_active]    = 10.0
    return w

cliff_role_arr = tr['cliff_role'].values
sample_weights = make_sample_weights(y_tr, cliff_role_arr)

print('Weight distribution:')
for w_val in sorted(np.unique(sample_weights)):
    n = (sample_weights == w_val).sum()
    print(f'  w={w_val:.1f}: {n:,} compounds')

Weight distribution:
  w=1.0: 2,775 compounds
  w=3.0: 1,269 compounds
  w=6.0: 67 compounds
  w=8.0: 28 compounds


## 3. Auxiliary cliff-role classifier (3-class LGBM)

In [5]:
# 3-class cliff role: map {-1, 0, 1} -> {0, 1, 2} for LGBM multiclass
cliff_label_map = {-1: 0, 0: 1, 1: 2}
cliff_labels_mc = np.array([cliff_label_map[c] for c in cliff_role_arr])

clf_params = dict(
    n_estimators=500, num_leaves=32, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_samples=10, n_jobs=4, verbose=-1,
    objective='multiclass', num_class=3,
)

oof_cliff_proba = np.zeros((len(tr), 3), dtype=np.float32)
fold_accs = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    clf = lgb.LGBMClassifier(**clf_params)
    clf.fit(X_tr[tr_idx], cliff_labels_mc[tr_idx])
    proba = clf.predict_proba(X_tr[va_idx])  # (n_val, 3)
    oof_cliff_proba[va_idx] = proba
    preds_cls = proba.argmax(1)
    acc = (preds_cls == cliff_labels_mc[va_idx]).mean()
    fold_accs.append(acc)
    print(f'  Fold {fold}: cliff-role accuracy = {acc:.3f}')

print(f'\nMean cliff-role accuracy: {np.mean(fold_accs):.3f}')

# Note: majority class (cliff_role==0) dominates — accuracy by itself is misleading;
# what matters is that cliff_active probability (class 2) correlates with true cliffs.
cliff_true_idx = cliff_labels_mc == 2
if cliff_true_idx.sum() > 0:
    avg_prob_on_true = oof_cliff_proba[cliff_true_idx, 2].mean()
    avg_prob_on_false = oof_cliff_proba[~cliff_true_idx, 2].mean()
    print(f'Avg cliff-active prob on true cliff-actives:  {avg_prob_on_true:.3f}')
    print(f'Avg cliff-active prob on non-cliff-actives:   {avg_prob_on_false:.3f}')

# Save OOF cliff probabilities for downstream ensembles
np.save(DATA_PROCESSED / 'oof_cliff_role_proba.npy', oof_cliff_proba)
print('Saved oof_cliff_role_proba.npy')

  Fold 0: cliff-role accuracy = 0.986


  Fold 1: cliff-role accuracy = 0.986


  Fold 2: cliff-role accuracy = 0.987


  Fold 3: cliff-role accuracy = 0.981


  Fold 4: cliff-role accuracy = 0.985

Mean cliff-role accuracy: 0.985
Avg cliff-active prob on true cliff-actives:  0.000
Avg cliff-active prob on non-cliff-actives:   0.001
Saved oof_cliff_role_proba.npy


## 4. Main model: cliff-weighted LGBM

In [6]:
# ── Cliff-weighted LGBM ───────────────────────────────────────────────────
oof_weighted = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(
        X_tr[tr_idx], y_tr[tr_idx],
        sample_weight=sample_weights[tr_idx],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        eval_set=[(X_tr[va_idx], y_tr[va_idx])],
    )
    oof_weighted[va_idx] = m.predict(X_tr[va_idx])

rae_weighted_overall = rae(y_tr, oof_weighted)

# Cliff-specific and non-cliff RAE
cliff_mask = cliff_role_arr != 0
noncliff_mask = ~cliff_mask

rae_cliff  = rae(y_tr[cliff_mask], oof_weighted[cliff_mask]) if cliff_mask.sum() > 5 else float('nan')
rae_nocliff = rae(y_tr[noncliff_mask], oof_weighted[noncliff_mask]) if noncliff_mask.sum() > 5 else float('nan')

print(f'Cliff-weighted LGBM:')
print(f'  OOF RAE (overall):    {rae_weighted_overall:.4f}')
print(f'  OOF RAE (cliff only): {rae_cliff:.4f}  (n={cliff_mask.sum()})')
print(f'  OOF RAE (non-cliff):  {rae_nocliff:.4f}  (n={noncliff_mask.sum()})')

Cliff-weighted LGBM:
  OOF RAE (overall):    0.5677
  OOF RAE (cliff only): 0.6794  (n=62)
  OOF RAE (non-cliff):  0.5705  (n=4077)


In [7]:
# ── Unweighted baseline for comparison ────────────────────────────────────
oof_unweighted = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**LGBM_PARAMS)
    m.fit(
        X_tr[tr_idx], y_tr[tr_idx],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
        eval_set=[(X_tr[va_idx], y_tr[va_idx])],
    )
    oof_unweighted[va_idx] = m.predict(X_tr[va_idx])

rae_uw_overall = rae(y_tr, oof_unweighted)
rae_uw_cliff   = rae(y_tr[cliff_mask], oof_unweighted[cliff_mask]) if cliff_mask.sum() > 5 else float('nan')
rae_uw_nocliff = rae(y_tr[noncliff_mask], oof_unweighted[noncliff_mask]) if noncliff_mask.sum() > 5 else float('nan')

print(f'Unweighted LGBM baseline:')
print(f'  OOF RAE (overall):    {rae_uw_overall:.4f}')
print(f'  OOF RAE (cliff only): {rae_uw_cliff:.4f}')
print(f'  OOF RAE (non-cliff):  {rae_uw_nocliff:.4f}')

print(f'\nDelta (weighted - unweighted):')
print(f'  Overall: {rae_weighted_overall - rae_uw_overall:+.4f}')
print(f'  Cliff:   {rae_cliff - rae_uw_cliff:+.4f}')

Unweighted LGBM baseline:
  OOF RAE (overall):    0.5599
  OOF RAE (cliff only): 0.6707
  OOF RAE (non-cliff):  0.5626

Delta (weighted - unweighted):
  Overall: +0.0078
  Cliff:   +0.0087


## 5. Multi-output variant: ridge meta-learner stacking cliff-role probabilities

In [8]:
# Stack: [oof_weighted, oof_cliff_proba] -> Ridge meta-learner
# This tests whether knowing the cliff-role probability helps calibrate pEC50 predictions.

meta_features = np.column_stack([oof_weighted, oof_cliff_proba])  # (N, 4)

oof_meta = np.full(len(y_tr), np.nan)
for fold, (tr_idx, va_idx) in enumerate(splits):
    ridge = Ridge(alpha=1.0)
    ridge.fit(meta_features[tr_idx], y_tr[tr_idx])
    oof_meta[va_idx] = ridge.predict(meta_features[va_idx])

rae_meta = rae(y_tr, oof_meta)
print(f'Ridge meta-learner (weighted LGBM + cliff-role proba): OOF RAE = {rae_meta:.4f}')
print(f'  vs cliff-weighted LGBM alone:                                  {rae_weighted_overall:.4f}')

# Choose best OOF for downstream
if rae_meta < rae_weighted_overall:
    oof_best = oof_meta.copy()
    best_model_name = 'ridge_meta'
    print('Best: ridge meta-learner')
else:
    oof_best = oof_weighted.copy()
    best_model_name = 'cliff_weighted_lgbm'
    print('Best: cliff-weighted LGBM')

Ridge meta-learner (weighted LGBM + cliff-role proba): OOF RAE = 0.5681
  vs cliff-weighted LGBM alone:                                  0.5677
Best: cliff-weighted LGBM


## 6. Final model + save outputs

In [9]:
# ── Final cliff-weighted model trained on all data ────────────────────────
final_model = lgb.LGBMRegressor(**LGBM_PARAMS)
final_model.fit(
    X_tr, y_tr,
    sample_weight=sample_weights,
    callbacks=[lgb.log_evaluation(-1)],
)
te_preds = final_model.predict(X_te)

# Clip to training range ± 0.5
lo_clip = float(y_tr.min()) - 0.5
hi_clip = float(y_tr.max()) + 0.5
te_preds = np.clip(te_preds, lo_clip, hi_clip)

# If meta-learner was best, apply cliff-role classifier on test too
if best_model_name == 'ridge_meta':
    clf_final = lgb.LGBMClassifier(**clf_params)
    cliff_labels_mc_full = np.array([cliff_label_map[c] for c in cliff_role_arr])
    clf_final.fit(X_tr, cliff_labels_mc_full)
    te_cliff_proba = clf_final.predict_proba(X_te)
    meta_te = np.column_stack([te_preds, te_cliff_proba])
    # Re-fit ridge on full train
    ridge_final = Ridge(alpha=1.0)
    ridge_final.fit(meta_features, y_tr)
    meta_train_full = np.column_stack([final_model.predict(X_tr), clf_final.predict_proba(X_tr)])
    # Note: use original oof_weighted + oof_cliff_proba for the full-data ridge input
    ridge_final.fit(meta_features, y_tr)
    te_preds_meta = ridge_final.predict(np.column_stack([te_preds, te_cliff_proba]))
    te_preds_meta = np.clip(te_preds_meta, lo_clip, hi_clip)
    te_preds_final = te_preds_meta
else:
    te_preds_final = te_preds

print(f'Test predictions — min={te_preds_final.min():.2f}  '
      f'median={np.median(te_preds_final):.2f}  max={te_preds_final.max():.2f}')

# ── Save OOF and submission ───────────────────────────────────────────────
np.save(DATA_PROCESSED / 'oof_cliff_weighted.npy', oof_best)
print(f'Saved oof_cliff_weighted.npy  (OOF RAE = {rae(y_tr, oof_best):.4f})')

sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'pEC50': te_preds_final,
})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out_path = SUBMISSIONS / '40_cliff_weighted_lgbm.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(sub['pEC50'].describe().round(3))

Test predictions — min=2.28  median=4.98  max=6.00
Saved oof_cliff_weighted.npy  (OOF RAE = 0.5677)
Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\40_cliff_weighted_lgbm.csv
count    513.000
mean       4.806
std        0.679
min        2.275
25%        4.475
50%        4.977
75%        5.302
max        6.000
Name: pEC50, dtype: float64


## Summary

| Model | OOF RAE (overall) | OOF RAE (cliff) | OOF RAE (non-cliff) |
|---|---|---|---|
| Unweighted LGBM | see above | see above | see above |
| **Cliff-weighted LGBM** | **see above** | **see above** | **see above** |
| Ridge meta (+ cliff proba) | see above | — | — |

**Saved:**
- `data/processed/oof_cliff_weighted.npy` — best OOF pEC50 predictions
- `data/processed/oof_cliff_role_proba.npy` — OOF cliff-role class probabilities (3 columns)
- `submissions/40_cliff_weighted_lgbm.csv`